# Feature Stores (Feast)

In [ ]:
# Day 37 — Feature Stores (Feast)
# One of the hardest problems in MLOps is TRAINING-SERVING SKEW:
#   - Data scientists compute features in Pandas for offline training
#   - Backend engineers re-implement those features in SQL/Python for online API
#   - Subtle differences creep in -> model accuracy drops in production!
#
# A Feature Store solves this by defining features ONCE and serving them to both:
#   1. Offline Store (Parquet, BigQuery, Snowflake) -> for batch training
#   2. Online Store  (Redis, SQLite, DynamoDB)     -> for low-latency API inference
#
# Feast is the #1 open-source Python feature store.
#
# Today we'll cover:
#   1. Core Feast concepts (Entity, FeatureView, FileSource)
#   2. Setting up a local Feast repo (feature_store.yaml)
#   3. Defining features in Python
#   4. Retrieving historical features (training) vs online features (serving)
#
# Install: pip install feast

### 1. Creating Sample Feature Data & Repo Config

In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta

os.makedirs("feast_demo/data", exist_ok=True)

# Create sample time-stamped server metrics (Parquet format)
now = datetime.utcnow()
df = pd.DataFrame({
    "server_id": ["web-01", "web-02", "db-01", "web-01", "web-02"],
    "event_timestamp": [
        now - timedelta(hours=2),
        now - timedelta(hours=2),
        now - timedelta(hours=2),
        now - timedelta(minutes=10),
        now - timedelta(minutes=10),
    ],
    "cpu_avg_5m": [45.2, 82.1, 60.5, 48.0, 91.3],
    "mem_avg_5m": [60.0, 78.5, 84.0, 61.2, 85.0],
    "req_per_sec": [120, 450, 80, 135, 520]
})
df.to_parquet("feast_demo/data/server_metrics.parquet", index=False)

# Create feature_store.yaml (local SQLite online store + File offline store)
store_yaml = '''project: server_health_features
registry: data/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
entity_key_serialization_version: 2
'''
with open("feast_demo/feature_store.yaml", "w") as f:
    f.write(store_yaml)

print("Created feast_demo/data/server_metrics.parquet and feature_store.yaml")

### 2. Defining Entities & FeatureViews

In [ ]:
# In Feast, you define your features declaratively in a Python file.

feature_defs = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

# 1. Entity: the primary key we join features on
server = Entity(name="server", join_keys=["server_id"])

# 2. Source: where the raw computed features live
server_metrics_source = FileSource(
    name="server_metrics_source",
    path="data/server_metrics.parquet",
    timestamp_field="event_timestamp",
)

# 3. FeatureView: logical group of features + TTL
server_metrics_fv = FeatureView(
    name="server_metrics",
    entities=[server],
    ttl=timedelta(days=1),
    schema=[
        Field(name="cpu_avg_5m", dtype=Float32),
        Field(name="mem_avg_5m", dtype=Float32),
        Field(name="req_per_sec", dtype=Int64),
    ],
    online=True,
    source=server_metrics_source,
)
'''

with open("feast_demo/features.py", "w") as f:
    f.write(feature_defs)

print("Created feast_demo/features.py")
print("CLI workflow:")
print("  cd feast_demo")
print("  feast apply                   # register entity & feature views")
print("  feast materialize-incremental $(date -u +%Y-%m-%dT%H:%M:%S)  # load into online store")

### 3. Offline (Training) vs Online (Serving) Retrieval

In [ ]:
# How training scripts and FastAPI endpoints consume Feast:

usage_example = '''from feast import FeatureStore
import pandas as pd
from datetime import datetime

store = FeatureStore(repo_path="feast_demo")

# 1. OFFLINE: Point-in-time join for training data (prevents data leakage!)
entity_df = pd.DataFrame({
    "server_id": ["web-01", "web-02"],
    "event_timestamp": [datetime.utcnow(), datetime.utcnow()]
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "server_metrics:cpu_avg_5m",
        "server_metrics:mem_avg_5m",
        "server_metrics:req_per_sec",
    ]
).to_df()

# 2. ONLINE: Low-latency lookup inside FastAPI /predict endpoint
online_features = store.get_online_features(
    features=[
        "server_metrics:cpu_avg_5m",
        "server_metrics:mem_avg_5m",
    ],
    entity_rows=[{"server_id": "web-01"}]
).to_dict()
print(online_features)
'''

print("Training & Serving code pattern:")
print(usage_example)

In [ ]:
# Practice Questions:
#
# 1. What is training-serving skew and how does a feature store prevent it?
# 2. Why does `get_historical_features()` require an `event_timestamp` column?
# 3. What is the difference between `feast apply` and `feast materialize`?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: ML Feature Registry for Server Health
# Run `feast apply` and `feast materialize-incremental` inside `feast_demo/`
# and verify you can query `web-01` from the online store.

# Exercise 2: Training-Serving Consistency Checker
# Write a Python script that fetches features for `web-02` using both
# `get_historical_features()` and `get_online_features()` and asserts they match.

# Exercise 3: Feature Engineering Pipeline
# Add a second FeatureView (`server_error_stats`) with 1h rolling error counts
# and combine both FeatureViews into a single `FeatureService`.